# 07 — Esperimento ad Alta Risoluzione $k=7$ ($128 \times 128$): Baseline $k=7$ vs Varianti Ottimizzate (`Opt 1` e `Opt 3b`)

## Obiettivo Scientifico del Notebook
Verificare sperimentalmente l'impatto del passaggio da $k=6$ ($64 \times 64 = 4.096$ celle) a **$k=7$ ($128 \times 128 = 16.384$ celle)** sulla Rete Siamese (`SiameseV3`), confrontando direttamente:
1. **`siamese_baseline_k7` (FCGR Baseline Lineare $k=7$, 1 canale)**:
   - Nella versione classica (`float32` + ciclo Python) richiederebbe **`3.662 GB`** di memoria e **`~377 secondi`** di calcolo.
   - Grazie all'equivalenza matematica esatta di `compute_fcgr_vectorized` e alla rappresentazione probabilistica `uint8` (`sum = 1.0`, errore $< 1.5 \times 10^{-5}$), la generiamo in **~6 secondi** occupando esattamente **`915.5 MB`** (lo stesso budget RAM della vecchia baseline $k=6$!), testando così la vera distribuzione lineare $k=7$ (sparsità mediana **`96.2%`**).
2. **`siamese_opt1_rc_logprob_k7` (Variante `Opt 1` a $k=7$, 1 canale, `915.5 MB`)**:
   - Unisce **Reverse-Complement** + **Normalizzazione `Log-Prob` (`sum = 1.0`)** a risoluzione $128 \times 128$ (**-75% RAM** rispetto ai `3.66 GB` in `float32`).
3. **`siamese_opt3b_enriched_ms_k567` (Variante `Opt 3b` Multi-Scala & Enriched a $k=7$, 3 canali, `2.75 GB` invece di `10.99 GB`)**:
   - **Canale 0**: `k=7 Forward Log-Prob` (`128x128`, `sum = 1.0`)
   - **Canale 1**: `k=7 O/E Motif Enrichment` (`128x128`, `sum = 1.0`, arricchimento sovra-casuale contro il rumore GC/AT)
   - **Canale 2**: `Contesto Frattale Multi-Scala k=5+6` (`32x32` e `64x64` allineati a `128x128`, `sum = 1.0`, che abbatte la sparsità spaziale dal **`96.2%` a `~35-40%`**!).

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import copy
import json
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from numpy.lib.format import open_memmap
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_fcgr_siamese')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

from src.fcgr import (
    load_fasta_sequences,
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_vectorized,
    normalize_fcgr_log_prob,
    compute_opt3_enriched_fcgr,
    quantize_fcgr_uint8,
    dequantize_fcgr_uint8_prob,
    plot_opt3_enriched_diagnostics,
    plot_siamese_pair_feedback,
)
from src.dataset import (
    RandomSiamesePairDataset,
    BalancedSiamesePairDataset,
    BalancedSemiHardCandidateDataset,
    SingleFCGRDataset,
)
from src.models import (
    SiameseV3,
    EuclideanContrastiveLoss,
    select_semihard_negatives,
    evaluate_pairwise,
    extract_embeddings,
    compute_topk_similarity_matrix,
)

print('PROJECT_ROOT :', PROJECT_ROOT)

## 1. Ispezione Visiva a $k=7$ ($128 \times 128$): Baseline Lineare (`96.2%` sparsa) vs `Opt 3b` Multi-Scala ($k=5,6,7$)
Verifichiamo su 1 Hotspot e 1 Stabile come cambia la matrice salendo a $k=7$ ($128 \times 128$) e quanto la nostra `Opt 3b` riduca la sparsità spaziale.

In [ ]:
VAL_POOL_PATH = PROCESSED_DIR / 'siamese_val_pair_pool.tsv'
val_sample_df = pd.read_csv(VAL_POOL_PATH, sep='\t', dtype={'id': str})
sample_hotspot_id = str(val_sample_df[val_sample_df['class_id'] == 1].iloc[0]['id'])
sample_stable_id = str(val_sample_df[val_sample_df['class_id'] == 0].iloc[0]['id'])
demo_seqs = load_fasta_sequences(FASTA_PATH, [sample_hotspot_id, sample_stable_id])

for sid, lbl in [(sample_hotspot_id, 'Hotspot (1)'), (sample_stable_id, 'Stable (0)')]:
    _ = plot_opt3_enriched_diagnostics(
        sequence=demo_seqs[sid],
        target_k=7,
        sample_id=sid,
        label_name=lbl,
        save_path=COMPARISON_DIR / f'k7_opt3_enriched_diagnostics_{sid}.png',
    )

## 2. Precomputazione Veloce delle 3 Cache a $k=7$ (`128 x 128`)
Generiamo per tutte le `60.000` sequenze:
1. `fcgr_k7_baseline_linear_u8.npy` (`915.5 MB` invece di `3.662 GB` in `float32`): distribuzione lineare esatta della Baseline $k=7$.
2. `fcgr_opt1_rc_logprob_k7_u8.npy` (`915.5 MB`): Reverse-Complement + `Log-Prob (sum=1)` a $k=7$.
3. `fcgr_opt3b_enriched_ms_k567_u8.npy` (`2.75 GB` invece di `10.99 GB` in `float32`): 3 canali (`Ch0=k7 Log-Prob`, `Ch1=k7 O/E Enrichment`, `Ch2=k5+6 Multi-Scale Context`).

In [ ]:
K = 7
SIDE = 1 << K  # 128
RANDOM_STATE = 42
N_CLASSES = 2
EMBEDDING_DIM = 128
EUCLIDEAN_MARGIN = 1.25

WARMUP_EPOCHS = 3
MAX_SEMIHARD_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4

BALANCED_BATCH_SIZE = 64
VAL_BATCH_SIZE = 128
VAL_PAIRS = 10_000
TEST_PAIRS = 10_000
N_NEGATIVE_CANDIDATES = 6

WARMUP_LR = 5e-4
SEMIHARD_LR = 1e-4
WEIGHT_DECAY = 1e-4

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('Device      :', DEVICE, f'({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else '')

TEST_POOL_PATH = PROCESSED_DIR / 'siamese_test_pair_pool.tsv'
FCGR_CACHE_DIR = PROCESSED_DIR / 'fcgr_cache'
FCGR_INDEX_PATH = FCGR_CACHE_DIR / 'fcgr_k6_index.tsv'

metadata = pd.read_csv(MANIFEST_PATH, sep='\t', dtype={'id': str})
metadata['class_id'] = metadata['class_id'].astype(int)
train_metadata = metadata[metadata['split_cluster'] == 'train'].copy().reset_index(drop=True)
val_metadata = pd.read_csv(VAL_POOL_PATH, sep='\t', dtype={'id': str})
val_metadata['class_id'] = val_metadata['class_id'].astype(int)
test_metadata = pd.read_csv(TEST_POOL_PATH, sep='\t', dtype={'id': str})
test_metadata['class_id'] = test_metadata['class_id'].astype(int)

fcgr_index = pd.read_csv(FCGR_INDEX_PATH, sep='\t', dtype={'id': str})
id_to_fcgr_row = dict(zip(fcgr_index['id'], fcgr_index['fcgr_row']))
ordered_ids = fcgr_index.sort_values('fcgr_row')['id'].tolist()
N_TOTAL_SAMPLES = len(ordered_ids)

K7_BASE_CACHE_PATH = FCGR_CACHE_DIR / 'fcgr_k7_baseline_linear_u8.npy'
K7_OPT1_CACHE_PATH = FCGR_CACHE_DIR / 'fcgr_opt1_rc_logprob_k7_u8.npy'
K7_OPT3B_CACHE_PATH = FCGR_CACHE_DIR / 'fcgr_opt3b_enriched_ms_k567_u8.npy'

need_b7 = not K7_BASE_CACHE_PATH.exists()
need_o1 = not K7_OPT1_CACHE_PATH.exists()
need_o3 = not K7_OPT3B_CACHE_PATH.exists()

if need_b7 or need_o1 or need_o3:
    print(f'Caricamento delle {N_TOTAL_SAMPLES:,} sequenze dal FASTA...')
    t_fa = time.perf_counter()
    seq_store = load_fasta_sequences(FASTA_PATH, ordered_ids)
    print(f'Sequenze caricate in {time.perf_counter() - t_fa:.1f}s')

if need_b7:
    print('\n1) Generazione Cache Baseline Lineare k=7 (128x128, 915.5 MB invece di 3.66 GB)...')
    tmp_b7 = FCGR_CACHE_DIR / 'tmp_k7_base_u8.npy'
    mm_b7 = open_memmap(tmp_b7, mode='w+', dtype=np.uint8, shape=(N_TOTAL_SAMPLES, SIDE, SIDE))
    t0 = time.perf_counter()
    for r_idx, sid in enumerate(ordered_ids):
        c7, _, _ = compute_fcgr_vectorized(seq_store[sid], K, return_distinct=True, include_reverse_complement=False)
        mm_b7[r_idx] = quantize_fcgr_uint8(normalize_fcgr(c7))
        if (r_idx + 1) % 15_000 == 0 or (r_idx + 1) == N_TOTAL_SAMPLES:
            el = time.perf_counter() - t0
            print(f'  [{r_idx+1:6,d}/{N_TOTAL_SAMPLES:,}] ({(r_idx+1)/max(el,1e-6):.0f} seq/s, {el:.1f}s)')
    mm_b7.flush()
    del mm_b7
    tmp_b7.replace(K7_BASE_CACHE_PATH)
    print(f'Cache Baseline k=7 completata in {time.perf_counter() - t0:.1f}s (vs ~377s col ciclo Python originale)!')
else:
    print('Cache Baseline k=7 già presente su disco.')

if need_o1:
    print('\n2) Generazione Cache Opt 1 k=7: Reverse-Complement + Log-Prob (128x128, 915.5 MB)...')
    tmp_o1 = FCGR_CACHE_DIR / 'tmp_k7_opt1_u8.npy'
    mm_o1 = open_memmap(tmp_o1, mode='w+', dtype=np.uint8, shape=(N_TOTAL_SAMPLES, SIDE, SIDE))
    t0 = time.perf_counter()
    for r_idx, sid in enumerate(ordered_ids):
        c7_rc, _, _ = compute_fcgr_vectorized(seq_store[sid], K, return_distinct=True, include_reverse_complement=True)
        mm_o1[r_idx] = quantize_fcgr_uint8(normalize_fcgr_log_prob(c7_rc))
        if (r_idx + 1) % 15_000 == 0 or (r_idx + 1) == N_TOTAL_SAMPLES:
            el = time.perf_counter() - t0
            print(f'  [{r_idx+1:6,d}/{N_TOTAL_SAMPLES:,}] ({(r_idx+1)/max(el,1e-6):.0f} seq/s, {el:.1f}s)')
    mm_o1.flush()
    del mm_o1
    tmp_o1.replace(K7_OPT1_CACHE_PATH)
    print(f'Cache Opt 1 k=7 completata in {time.perf_counter() - t0:.1f}s!')
else:
    print('Cache Opt 1 k=7 già presente su disco.')

if need_o3:
    print('\n3) Generazione Cache Opt 3b k=7: Enriched O/E + Multi-Scala k=5,6,7 (3x128x128, 2.75 GB invece di 10.99 GB)...')
    tmp_o3 = FCGR_CACHE_DIR / 'tmp_k7_opt3b_u8.npy'
    mm_o3 = open_memmap(tmp_o3, mode='w+', dtype=np.uint8, shape=(N_TOTAL_SAMPLES, 3, SIDE, SIDE))
    t0 = time.perf_counter()
    for r_idx, sid in enumerate(ordered_ids):
        st7, _ = compute_opt3_enriched_fcgr(seq_store[sid], target_k=K)
        mm_o3[r_idx] = quantize_fcgr_uint8(st7)
        if (r_idx + 1) % 15_000 == 0 or (r_idx + 1) == N_TOTAL_SAMPLES:
            el = time.perf_counter() - t0
            print(f'  [{r_idx+1:6,d}/{N_TOTAL_SAMPLES:,}] ({(r_idx+1)/max(el,1e-6):.0f} seq/s, {el:.1f}s)')
    mm_o3.flush()
    del mm_o3
    tmp_o3.replace(K7_OPT3B_CACHE_PATH)
    print(f'Cache Opt 3b k=7 completata in {time.perf_counter() - t0:.1f}s!')
else:
    print('Cache Opt 3b k=7 già presente su disco.')

## 3. Pipeline di Training e Validazione Downstream su `SiameseV3` ($k=7$)
Grazie a `AdaptiveAvgPool2d((4, 4))`, `SiameseV3` elabora nativamente le matrici $128 \times 128$ mantenendo **esattamente lo stesso numero di parametri** (`807.264` per 1 canale, `807.840` per 3 canali).

In [ ]:
WARMUP_BATCHES_PER_EPOCH = math.ceil(len(train_metadata) / BALANCED_BATCH_SIZE)
WARMUP_PAIRS_PER_EPOCH = WARMUP_BATCHES_PER_EPOCH * BALANCED_BATCH_SIZE
MINING_BATCHES_PER_EPOCH = math.ceil((WARMUP_PAIRS_PER_EPOCH / 2) / BALANCED_BATCH_SIZE)

def train_random_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    model.train()
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances = [], []
    start_time = time.perf_counter()
    for batch in loader:
        x1 = batch['x1'].to(DEVICE, non_blocking=True)
        x2 = batch['x2'].to(DEVICE, non_blocking=True)
        target = batch['target'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            z1, z2 = model(x1, x2)
        loss, distances = criterion(z1, z2, target)
        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        bs = target.shape[0]
        total_loss += loss.item() * bs
        total_samples += bs
        all_targets.append(target.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())
    targets = np.concatenate(all_targets)
    distances = np.concatenate(all_distances)
    return {'loss': total_loss / total_samples, 'auc': float(roc_auc_score(targets, -distances)), 'seconds': time.perf_counter() - start_time}

def train_dynamic_semihard_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances, semihard_fractions = [], [], []
    start_time = time.perf_counter()
    for batch in loader:
        anchor = batch['anchor'].to(DEVICE, non_blocking=True)
        positive = batch['positive'].to(DEVICE, non_blocking=True)
        negatives = batch['negatives'].to(DEVICE, non_blocking=True)
        selected_negative, mining_stats = select_semihard_negatives(
            model=model, anchor=anchor, positive=positive, negatives=negatives,
            margin=EUCLIDEAN_MARGIN, amp_enabled=AMP_ENABLED,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        bs = anchor.shape[0]
        train_images = torch.cat([anchor, positive, selected_negative], dim=0)
        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            embeddings = model.encoder(train_images)
        z_a, z_p, z_n = embeddings[:bs], embeddings[bs:2*bs], embeddings[2*bs:]
        z1 = torch.cat([z_a, z_a], dim=0)
        z2 = torch.cat([z_p, z_n], dim=0)
        targets = torch.cat([torch.ones(bs, device=DEVICE), torch.zeros(bs, device=DEVICE)], dim=0)
        loss, distances = criterion(z1, z2, targets)
        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        n_pairs = targets.shape[0]
        total_loss += loss.detach().item() * n_pairs
        total_samples += n_pairs
        all_targets.append(targets.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())
        semihard_fractions.append(mining_stats['semihard_fraction'])
    targets_np = np.concatenate(all_targets)
    distances_np = np.concatenate(all_distances)
    return {'loss': total_loss / total_samples, 'auc': float(roc_auc_score(targets_np, -distances_np)), 'semihard_fraction': float(np.mean(semihard_fractions)), 'seconds': time.perf_counter() - start_time}

def run_siamese_experiment(exp_name: str, fcgr_cache_path: Path, in_channels: int, k_order: int = 7):
    print('=' * 95)
    print(f'AVVIO ESPERIMENTO: {exp_name} (k={k_order}, in_channels={in_channels})')
    print('=' * 95)
    out_dir = ARTIFACTS_DIR / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)
    fcgr_mm = np.load(fcgr_cache_path, mmap_mode='r')
    cache_mb = fcgr_cache_path.stat().st_size / (1024 ** 2)
    print(f'Cache aperta: {fcgr_cache_path.name} | shape={fcgr_mm.shape} | dtype={fcgr_mm.dtype} | {cache_mb:.1f} MB')

    set_seed(RANDOM_STATE)
    warmup_ds = BalancedSiamesePairDataset(train_metadata, fcgr_mm, id_to_fcgr_row, WARMUP_BATCHES_PER_EPOCH, BALANCED_BATCH_SIZE, 0.50, RANDOM_STATE)
    mining_ds = BalancedSemiHardCandidateDataset(train_metadata, fcgr_mm, id_to_fcgr_row, MINING_BATCHES_PER_EPOCH, BALANCED_BATCH_SIZE, N_NEGATIVE_CANDIDATES, RANDOM_STATE)
    val_ds = RandomSiamesePairDataset(val_metadata, fcgr_mm, id_to_fcgr_row, VAL_PAIRS, 0.50, RANDOM_STATE + 50_000)
    test_ds = RandomSiamesePairDataset(test_metadata, fcgr_mm, id_to_fcgr_row, TEST_PAIRS, 0.50, RANDOM_STATE + 60_000)

    warmup_loader = DataLoader(warmup_ds, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    mining_loader = DataLoader(mining_ds, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    test_loader = DataLoader(test_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)

    model = SiameseV3(embedding_dim=EMBEDDING_DIM, in_channels=in_channels).to(DEVICE)
    criterion = EuclideanContrastiveLoss(margin=EUCLIDEAN_MARGIN)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Parametri trainabili SiameseV3: {n_params:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=WARMUP_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

    print('\n--- FASE 1: RANDOM-PAIR WARM-UP ---')
    for epoch in range(1, WARMUP_EPOCHS + 1):
        tr = train_random_epoch(model, warmup_loader, warmup_ds, criterion, optimizer, scaler, epoch)
        va = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
        print(f"Warm-up {epoch:02d}/{WARMUP_EPOCHS} | train AUC {tr['auc']:.4f} | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s")

    print('\n--- FASE 2: DYNAMIC SEMI-HARD NEGATIVE MINING ---')
    ckpt_path = out_dir / 'best_model.pt'
    optimizer = torch.optim.AdamW(model.parameters(), lr=SEMIHARD_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_metrics = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
    best_auc = best_metrics['auc']
    best_epoch = 0
    epochs_no_improve = 0
    history = []
    torch.save({'epoch': 0, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': best_metrics}, ckpt_path)

    for epoch in range(1, MAX_SEMIHARD_EPOCHS + 1):
        tr = train_dynamic_semihard_epoch(model, mining_loader, mining_ds, criterion, optimizer, scaler, WARMUP_EPOCHS + epoch)
        va = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
        history.append({'epoch': epoch, 'train_auc': tr['auc'], 'semihard_fraction': tr['semihard_fraction'], **va})
        if va['auc'] > best_auc + MIN_DELTA:
            best_auc = va['auc']
            best_epoch = epoch
            best_metrics = copy.deepcopy(va)
            epochs_no_improve = 0
            torch.save({'epoch': epoch, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': best_metrics}, ckpt_path)
            marker = ' <-- BEST'
        else:
            epochs_no_improve += 1
            marker = ''
        print(f"Epoch {epoch:02d}/{MAX_SEMIHARD_EPOCHS} | train AUC {tr['auc']:.4f} | semi-hard {tr['semihard_fraction']*100:.1f}% | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s{marker}")
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print('Early stopping attivato.')
            break

    pd.DataFrame(history).to_csv(out_dir / 'training_history.tsv', sep='\t', index=False)
    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt['model_state_dict'])
    model.eval()

    @torch.no_grad()
    def collect_dists(loader):
        t_all, d_all = [], []
        for b in loader:
            x1 = b['x1'].to(DEVICE, non_blocking=True)
            x2 = b['x2'].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
                z1, z2 = model(x1, x2)
            d = torch.linalg.vector_norm(z1.float() - z2.float(), ord=2, dim=1)
            t_all.append(b['target'].numpy())
            d_all.append(d.cpu().numpy())
        return np.concatenate(t_all), np.concatenate(d_all)

    val_t, val_d = collect_dists(val_loader)
    test_t, test_d = collect_dists(test_loader)
    thrs = np.linspace(val_d.min(), val_d.max(), 400)
    best_thr = float(thrs[int(np.argmax([f1_score(val_t, (val_d <= t).astype(int)) for t in thrs]))])
    test_preds_p = (test_d <= best_thr).astype(int)
    test_scores_p = -test_d
    pos_d, neg_d = test_d[test_t == 1], test_d[test_t == 0]
    gap_d = float(neg_d.mean() - pos_d.mean())
    d_prime_test = float(gap_d / np.sqrt(0.5 * (pos_d.var() + neg_d.var()) + 1e-12))

    pairwise_metrics = {
        'pairwise_roc_auc': float(roc_auc_score(test_t, test_scores_p)),
        'pairwise_pr_auc': float(average_precision_score(test_t, test_scores_p)),
        'pairwise_f1': float(f1_score(test_t, test_preds_p)),
        'pairwise_macro_f1': float(f1_score(test_t, test_preds_p, average='macro')),
        'pairwise_precision': float(precision_score(test_t, test_preds_p)),
        'pairwise_recall': float(recall_score(test_t, test_preds_p)),
        'pairwise_accuracy': float(accuracy_score(test_t, test_preds_p)),
        'd_pos_mean': float(pos_d.mean()),
        'd_neg_mean': float(neg_d.mean()),
        'distance_gap': gap_d,
        'd_prime': d_prime_test,
        'threshold': best_thr,
    }

    train_emb_loader = DataLoader(SingleFCGRDataset(train_metadata, fcgr_mm, id_to_fcgr_row), batch_size=256, shuffle=False)
    test_emb_loader = DataLoader(SingleFCGRDataset(test_metadata, fcgr_mm, id_to_fcgr_row), batch_size=256, shuffle=False)
    train_emb, train_lbl = extract_embeddings(model, train_emb_loader, DEVICE, AMP_ENABLED)
    test_emb, test_lbl = extract_embeddings(model, test_emb_loader, DEVICE, AMP_ENABLED)

    ref_per_class = min(5000, int((train_lbl == 0).sum()), int((train_lbl == 1).sum()))
    rng = np.random.default_rng(RANDOM_STATE)
    ref_idx = np.concatenate([
        rng.choice(np.where(train_lbl == 0)[0], size=ref_per_class, replace=False),
        rng.choice(np.where(train_lbl == 1)[0], size=ref_per_class, replace=False),
    ])
    sim_mat = compute_topk_similarity_matrix(test_emb, train_emb[ref_idx], train_lbl[ref_idx], n_classes=2, top_k=20)
    hotspot_score = sim_mat[:, 1] - sim_mat[:, 0]
    test_pred_cls = np.argmax(sim_mat, axis=1)

    cls_metrics = {
        'cls_roc_auc': float(roc_auc_score(test_lbl, hotspot_score)),
        'cls_pr_auc': float(average_precision_score(test_lbl, hotspot_score)),
        'cls_f1': float(f1_score(test_lbl, test_pred_cls)),
        'cls_macro_f1': float(f1_score(test_lbl, test_pred_cls, average='macro')),
        'cls_precision': float(precision_score(test_lbl, test_pred_cls)),
        'cls_recall': float(recall_score(test_lbl, test_pred_cls)),
        'cls_balanced_accuracy': float(balanced_accuracy_score(test_lbl, test_pred_cls)),
    }

    summary = {
        **pairwise_metrics,
        **cls_metrics,
        'k': k_order,
        'in_channels': in_channels,
        'cache_mb': round(cache_mb, 2),
        'train_samples': len(train_metadata),
        'best_epoch': best_epoch,
    }
    pd.DataFrame([summary]).to_csv(out_dir / f'{exp_name}_final_metrics.csv', index=False)
    with open(out_dir / f'{exp_name}_final_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=4)

    print(f'\n=== RISULTATI TEST SET (TUTTE LE METRICHE): {exp_name} ===')
    pd.set_option('display.max_columns', None)
    display(pd.DataFrame([summary]))
    print(classification_report(test_lbl, test_pred_cls, target_names=['Stable (0)', 'Hotspot (1)'], digits=4))
    return model, test_loader, summary

## 4. Esperimento 1 ($k=7$): `siamese_baseline_k7` (FCGR Baseline Lineare $k=7$, Sparsità `96.2%`)

In [ ]:
model_b7, test_loader_b7, summary_b7 = run_siamese_experiment(
    exp_name='siamese_baseline_k7',
    fcgr_cache_path=K7_BASE_CACHE_PATH,
    in_channels=1,
    k_order=7,
)

## 5. Esperimento 2 ($k=7$): `siamese_opt1_rc_logprob_k7` (Reverse-Complement + `Log-Prob` `sum=1`, 1 Canale, `915.5 MB`)

In [ ]:
model_o1_7, test_loader_o1_7, summary_o1_7 = run_siamese_experiment(
    exp_name='siamese_opt1_rc_logprob_k7',
    fcgr_cache_path=K7_OPT1_CACHE_PATH,
    in_channels=1,
    k_order=7,
)

## 6. Esperimento 3 ($k=7$): `siamese_opt3b_enriched_ms_k567` (Enriched O/E + Multi-Scala $k=5,6,7$, 3 Canali, `2.75 GB` invece di `10.99 GB`)

In [ ]:
model_o3_7, test_loader_o3_7, summary_o3_7 = run_siamese_experiment(
    exp_name='siamese_opt3b_enriched_ms_k567',
    fcgr_cache_path=K7_OPT3B_CACHE_PATH,
    in_channels=3,
    k_order=7,
)

## 7. Confronto Globale $k=6$ vs $k=7$ (Tutti gli 8 Esperimenti) e Feedback Visivo Coppie $k=7$

In [ ]:
pd.set_option('display.max_columns', None)
metric_files = (
    sorted(ARTIFACTS_DIR.glob('siamese_baseline_*/*_final_metrics.csv'))
    + sorted(ARTIFACTS_DIR.glob('siamese_opt*/*_final_metrics.csv'))
)
dfs = []
for f in metric_files:
    df_m = pd.read_csv(f)
    df_m.insert(0, 'experiment', f.parent.name)
    dfs.append(df_m)
all_comp_df = pd.concat(dfs, ignore_index=True)
all_comp_df.to_csv(COMPARISON_DIR / 'siamese_binary_comparison_table.csv', index=False)
cols_focus = [
    'experiment', 'k', 'in_channels', 'cache_mb', 'cls_roc_auc', 'cls_pr_auc', 'cls_f1', 'cls_balanced_accuracy',
    'pairwise_roc_auc', 'pairwise_pr_auc', 'd_pos_mean', 'd_neg_mean', 'distance_gap', 'd_prime', 'best_epoch'
]
print('=== TABELLA COMPARATIVA GLOBALE (k=6 vs k=7 — BASELINE vs OTTIMIZZATE) ===')
display(all_comp_df[[c for c in cols_focus if c in all_comp_df.columns]])

test_batch_7 = next(iter(test_loader_o3_7))
thr_7 = summary_o3_7['threshold']
with torch.no_grad():
    z1_b, z2_b = model_o3_7(test_batch_7['x1'].to(DEVICE), test_batch_7['x2'].to(DEVICE))
    dists_b = torch.linalg.vector_norm(z1_b.float() - z2_b.float(), ord=2, dim=1).cpu().numpy()
targets_b = test_batch_7['target'].numpy()

pos_indices = np.where((targets_b == 1) & (dists_b <= thr_7))[0]
neg_indices = np.where((targets_b == 0) & (dists_b > thr_7))[0]
pos_idx = int(pos_indices[0]) if len(pos_indices) > 0 else int(np.where(targets_b == 1)[0][0])
neg_idx = int(neg_indices[0]) if len(neg_indices) > 0 else int(np.where(targets_b == 0)[0][0])

for idx, desc in [(pos_idx, 'k=7 Opt3b Positiva'), (neg_idx, 'k=7 Opt3b Negativa')]:
    plot_siamese_pair_feedback(
        x1=test_batch_7['x1'][idx],
        x2=test_batch_7['x2'][idx],
        target=float(targets_b[idx]),
        label1=f'{desc} (Ramo 1)',
        label2=f'{desc} (Ramo 2)',
        embedding_distance=float(dists_b[idx]),
        threshold=thr_7,
    )